In [ ]:
import numpy as np
import metpy.calc as mpcalc
from metpy.units import units

def compute_pv_pressure_full(T_field, U_field, V_field, plev, lat, lon, target_pressure):
    """
    计算完整3D场上的潜在涡度（PV）。
    
    参数：
      T_field, U_field, V_field：3D 数组，形状 (nlev, nlat, nlon)，单位分别为 [K]、[m/s]、[m/s]。
      plev：1D 数组，代表各层压力，单位为 Pa。
      lat, lon：1D 数组，经纬度（单位：度）。
      target_pressure：目标压力，单位 Pa，用于从3D PV场中选取对应层。
    
    返回：
      2D 数组（nlat, nlon），单位为 PVU（1 PVU = 1e-6 K·m²/(kg·s)）。
    """
    print("=== Inside compute_pv_pressure_full ===")
    print("T_field shape:", T_field.shape)
    print("U_field shape:", U_field.shape)
    print("V_field shape:", V_field.shape)
    print("plev shape:", plev.shape)
    print("lat shape:", np.array(lat).shape)
    print("lon shape:", np.array(lon).shape)
    
    # 将 plev 重塑为 (nlev, 1, 1) 用于广播
    plev_reshaped = plev[:, None, None]
    
    # 计算潜热温度 theta
    theta = mpcalc.potential_temperature(plev_reshaped * units.Pa, T_field * units.K)
    print("theta shape:", theta.shape)
    
    # 构造 2D 的经纬度网格
    lon2d, lat2d = np.meshgrid(lon, lat)
    print("lon2d shape:", lon2d.shape, "lat2d shape:", lat2d.shape)
    
    # 使用 MetPy 计算水平网格间距，返回的 dx 形状为 (nlat, nlon-1)，dy 为 (nlat-1, nlon)
    dx_array, dy_array = mpcalc.lat_lon_grid_deltas(lon2d, lat2d)
    # 这里采用网格间距的均值作为标量（这样可以避免因局部尺寸不匹配导致的广播错误）
    dx_scalar = np.mean(dx_array)
    dy_scalar = np.mean(dy_array)
    print("dx_scalar:", dx_scalar, "dy_scalar:", dy_scalar)
    
    # 将2D经纬度转换为弧度
    lat2d_rad = np.deg2rad(lat2d)
    
    # 调用 MetPy 的 PV 计算函数，此处传入标量 dx 和 dy
    pv_field = mpcalc.potential_vorticity_baroclinic(
        theta, 
        plev * units.Pa,
        U_field * units('m/s'),
        V_field * units('m/s'),
        dx_scalar,
        dy_scalar,
        latitude=lat2d_rad,
        x_dim=2,
        y_dim=1,
        vertical_dim=0
    )
    print("pv_field shape:", pv_field.shape)
    
    # 如果 pv_field 是3D，则选取离 target_pressure 最近的那一层
    if pv_field.ndim == 3:
        idx = np.argmin(np.abs(plev - target_pressure))
        print("Selected vertical index:", idx)
        pv_target = pv_field[idx, :, :]
    else:
        pv_target = pv_field
    
    print("Returning pv_target with shape:", pv_target.shape)
    # 先转换到基础单位（K*m**2/(kg*s)），再手动乘以 1e-6 以获得 PVU
    pv_converted = pv_target.to('K*m**2/(kg*s)').magnitude * 1e-6
    return pv_converted

if __name__ == "__main__":
    # 固定随机种子以便重现
    np.random.seed(0)
    
    # 定义数据维度
    nlev = 23
    nlat = 96
    nlon = 144
    
    # 构造压力层，单位 Pa，假设从表层高压到顶层低压
    plev = np.linspace(100000, 10000, nlev)
    
    # 构造经纬度数据（纬度：60° ~ 90°，经度：-180° ~ 180°）
    lat = np.linspace(60, 90, nlat)
    lon = np.linspace(-180, 180, nlon)
    
    # 构造模拟温度场 T（单位：K），假设随高度递减并加上随机扰动
    T_base = np.linspace(280, 210, nlev)  # 例如从 280 K 递减到 210 K
    T_field = np.empty((nlev, nlat, nlon))
    for i in range(nlev):
        T_field[i, :, :] = T_base[i] + 5 * np.random.randn(nlat, nlon)
    
    # 构造模拟风场 U, V（单位：m/s）
    U_field = 10 + 2 * np.random.randn(nlev, nlat, nlon)
    V_field = 10 + 2 * np.random.randn(nlev, nlat, nlon)
    
    # 指定目标压力，此处随便指定一个，例如 50000 Pa（约 500 hPa）
    target_pressure = 50000  # Pa
    
    # 调用 PV 计算函数
    pv_result = compute_pv_pressure_full(T_field, U_field, V_field, plev, lat, lon, target_pressure)
    
    print("Resulting PV shape:", pv_result.shape)
    print("Resulting PV (sample):")
    print(pv_result)
